# Mixing and CP violation

$D^0$–$\bar{D}^0$ mixing and CP violation in $D^0 \to K^0_S \pi^+ \pi^-$, parametrised
as in the bin-flip method
([LHCb, PRL 127 (2021) 111801](https://arxiv.org/abs/2106.03744)) by

$$
z_\mathrm{CP} \pm \Delta z \equiv -\left(q/p\right)^{\pm 1} (y + i x),
\qquad z_\mathrm{CP} = -(y_\mathrm{CP} + i x_\mathrm{CP}),
\quad \Delta z = -(\Delta y + i \Delta x),
$$

instead of $x$, $y$, $|q/p|$ and $\phi = \arg(q/p)$. Without CP violation in mixing
($q/p = 1$), $x_\mathrm{CP} = x$, $y_\mathrm{CP} = y$ and $\Delta x = \Delta y = 0$.
These parameters are less correlated than $|q/p|$ and $\phi$, have Gaussian
uncertainties also when CP violation is small, and are the ones that a bin-flip
analysis measures, so that the two methods below can be compared directly.

Toys of $D^0$ and $\bar{D}^0$ are generated with the Belle 2018 amplitude model and
fitted (i) with the full time-dependent amplitude model, simultaneously for both
flavours, and (ii) with the model-independent bin-flip method. The mixing and CP
violation are exaggerated so that small toys suffice.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Math
from tensorwaves.data import IntensityDistributionGenerator
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.estimator import UnbinnedNLL
from tensorwaves.function.sympy import create_parametrized_function
from tensorwaves.optimizer import Minuit2

from aman_examples.binflip import (
    binflip_bin,
    binflip_ratio,
    fit_binflip,
    strong_phase_parameters,
    time_binned_ratios,
)
from aman_examples.d02kspipi import formulate_model
from aman_examples.estimator import SimultaneousEstimator
from aman_examples.mixing import (
    amplitude_transformer,
    formulate_mixing_model_zcp,
    mixing_from_zcp,
    zcp_from_mixing,
)
from aman_examples.phasespace import DalitzDecayTimeGenerator, DalitzPlotGenerator

warnings.filterwarnings("ignore", category=RuntimeWarning)

## Model

The true values are $x = 0.03$, $y = 0.025$, $|q/p| = 0.8$ and $\phi = -0.3$, converted
with `zcp_from_mixing` (and back with `mixing_from_zcp`):

In [ ]:
truth = zcp_from_mixing(x=0.03, y=0.025, q_over_p=0.8, phase=-0.3)
NAMES = tuple(truth)
print({k: round(float(v), 5) for k, v in truth.items()})
print({k: round(float(v), 5) for k, v in mixing_from_zcp(*truth.values()).items()})

`formulate_mixing_model_zcp` builds the rate of `formulate_mixing_model` with $x$, $y$
and $q/p$ written in terms of the new parameters,
$y + i x = \sqrt{(y_\mathrm{CP} + i x_\mathrm{CP})^2 - (\Delta y + i \Delta x)^2}$ and
$q/p = -(z_\mathrm{CP} + \Delta z) / (y + i x)$. The rate of a $\bar{D}^0$
(`initial_d0=False`) is evaluated at the same Dalitz-plot point, with the roles of
$A_f$ and $\bar{A}_f$ exchanged:

In [ ]:
rates = {
    "D0": formulate_mixing_model_zcp(*truth.values(), initial_d0=True),
    "D0bar": formulate_mixing_model_zcp(*truth.values(), initial_d0=False),
}
Math(sp.latex(rates["D0"].intensity))

In [ ]:
dalitz_model = formulate_model()
masses = [
    dalitz_model.parameter_defaults[dalitz_model.masses[k]]
    for k in ("pi", "KS", "pi", "D")
]
transformer = amplitude_transformer(dalitz_model)
functions = {
    flavour: create_parametrized_function(
        rate.intensity.doit(), rate.parameter_defaults, backend="jax"
    )
    for flavour, rate in rates.items()
}

## Toys

200k events of each flavour; decay times in units of $\tau$. The normalisation sample
is uniform in the Dalitz plot, with exponential decay times (the rate is written
relative to $e^{-t/\tau}$).

In [ ]:
generator = DalitzDecayTimeGenerator(masses)
rng = NumpyUniformRNG(seed=11)
phsp = transformer(generator.generate(1_000_000, rng))
data = {
    flavour: IntensityDistributionGenerator(
        generator, function, domain_transformer=transformer, bunch_size=200_000
    ).generate(200_000, rng)
    for flavour, function in functions.items()
}

## Amplitude fit

The two unbinned likelihoods share the parameters and are minimised together with
`SimultaneousEstimator`, which sums the estimators and their gradients. The amplitude
model is fixed: only the four mixing parameters are free.

In [ ]:
estimator = SimultaneousEstimator(
    *(
        UnbinnedNLL(functions[f], transformer(data[f]), phsp, backend="jax")
        for f in functions
    )
)
result = Minuit2().optimize(
    estimator, {"x_CP": 0.01, "y_CP": 0.01, "Delta_x": 0.0, "Delta_y": 0.0}
)
assert result.minimum_valid
amplitude_fit = {
    k: (result.parameter_values[k], result.parameter_errors[k]) for k in NAMES
}

## Bin-flip fit

The Dalitz plot is divided into the 8+8 bins of the BaBar 2008 "equal
$\Delta\delta_D$" binning (`binflip_bin`), and the data into 10 decay-time bins. For
each pair of bins $\pm b$, the ratio $R_b(t) = N_{+b}(t)/N_{-b}(t)$ is, to second order
in the mixing parameters,

$$
R_b^\pm = \frac{
  r_b \left(1 + \frac{\langle t^2\rangle}{4} \mathrm{Re}\,w^2\right)
  + \sqrt{r_b}\,\langle t\rangle\, \mathrm{Re}\left[X_b (z_\mathrm{CP} \pm \Delta z)\right]
  + \frac{\langle t^2\rangle}{4} \left|z_\mathrm{CP} \pm \Delta z\right|^2
}{
  1 + \frac{\langle t^2\rangle}{4} \mathrm{Re}\,w^2
  + \sqrt{r_b}\,\langle t\rangle\, \mathrm{Re}\left[X_b^* (z_\mathrm{CP} \pm \Delta z)\right]
  + r_b \frac{\langle t^2\rangle}{4} \left|z_\mathrm{CP} \pm \Delta z\right|^2
},
$$

with $w^2 = z_\mathrm{CP}^2 - \Delta z^2$, $X_b = c_b - i s_b$ the amplitude-weighted
strong-phase parameters, $r_b = F_{+b}/F_{-b}$ the ratio of the time-integrated $D^0$
yields, and $\pm$ for $D^0$ and $\bar{D}^0$ (whose Dalitz-plot coordinates are swapped
before binning). In a measurement $c_b$, $s_b$ come from charm-threshold data (CLEO,
BESIII); here they are computed from the model, on a Dalitz-plot sample symmetrised
under $s_- \leftrightarrow s_+$. The $r_b$ are free.

In [ ]:
half = DalitzPlotGenerator(masses).generate(1_000_000, NumpyUniformRNG(seed=2))
uniform = {
    "m2_kspim": np.concatenate([half["m2_kspim"], half["m2_kspip"]]),
    "m2_kspip": np.concatenate([half["m2_kspip"], half["m2_kspim"]]),
}
uniform["t"] = np.zeros(len(uniform["m2_kspim"]))
amplitudes = transformer(uniform)
strong = strong_phase_parameters(
    np.asarray(amplitudes["A_f"]),
    np.asarray(amplitudes[R"\bar{A}_f"]),
    binflip_bin(uniform["m2_kspim"], uniform["m2_kspip"]),
)
x_b = strong["c"] - 1j * strong["s"]
r_model = strong["F_plus"] / strong["F_minus"]

times = np.concatenate([data[f]["t"] for f in data])
edges = np.quantile(times, np.linspace(0, 1, 11))
edges[-1] += 1e-9
ratios = {
    "D0": time_binned_ratios(
        data["D0"]["t"],
        binflip_bin(data["D0"]["m2_kspim"], data["D0"]["m2_kspip"]),
        edges,
    ),
    "D0bar": time_binned_ratios(
        data["D0bar"]["t"],
        binflip_bin(data["D0bar"]["m2_kspip"], data["D0bar"]["m2_kspim"]),
        edges,
    ),
}
minuit = fit_binflip(ratios, x_b, r_model)
assert minuit.valid
binflip_fit = {k: (minuit.values[k], minuit.errors[k]) for k in NAMES}
ndf = 2 * 8 * (len(edges) - 1) - len(minuit.parameters)
print(f"chi2/ndf = {minuit.fval:.1f}/{ndf}")

The ratios and the fitted $R_b^\pm(t)$; the difference between $D^0$ (filled) and
$\bar{D}^0$ (open) is the CP violation, $\Delta z$:

In [ ]:
fitted = {k: minuit.values[k] for k in minuit.parameters}
z_cp = -(fitted["y_CP"] + 1j * fitted["x_CP"])
dz = -(fitted["Delta_y"] + 1j * fitted["Delta_x"])
r_fit = np.array([fitted[f"r_{b}"] for b in range(1, 9)])
t_curve = np.linspace(0, edges[-2] + 1, 200)
fig, axes = plt.subplots(2, 4, figsize=(12, 5.5), sharex=True, layout="constrained")
for flavour, sign, marker in (("D0", 1, "o"), ("D0bar", -1, "s")):
    ratio = ratios[flavour]
    label = R"$D^0$" if sign > 0 else R"$\bar{D}^0$"
    curve = binflip_ratio(
        t_curve, t_curve**2, r_fit, x_b, z_cp + sign * dz, z_cp**2 - dz**2
    )
    for b, ax in enumerate(axes.flat):
        color = "C0" if sign > 0 else "C3"
        ax.errorbar(
            ratio["t_mean"],
            ratio["ratio"][b] / r_fit[b],
            ratio["error"][b] / r_fit[b],
            fmt=marker,
            ms=3.5,
            color=color,
            mfc=color if sign > 0 else "white",
            label=label,
        )
        ax.plot(t_curve, curve[b] / r_fit[b], color=color)
        ax.set_title(f"bin {b + 1}")
for ax in axes[1]:
    ax.set_xlabel(R"$t/\tau$")
for ax in axes[:, 0]:
    ax.set_ylabel(R"$R_b(t)/r_b$")
axes[0, 0].legend()
plt.show()

## Comparison

Both fits recover the true values. The bin-flip method does not use the amplitude
model (apart from $c_b$, $s_b$, which would be measured), at the price of larger
uncertainties. These are largest for $y_\mathrm{CP}$ and $\Delta y$: in the ratios
$R_b$, the leading effect of $y_\mathrm{CP}$ is a change of the $D^0$ yields common to
the two bins of a pair, which cancels, and the free $r_b$ absorb part of the rest. The
amplitude fit also uses the variation of the strong phase within each bin, which the bin
averages $c_b$, $s_b$ dilute.

In [ ]:
print(
    f"{'':8s} {'true':>9s} {'amplitude fit':>21s} {'pull':>6s}  {'bin-flip fit':>21s} {'pull':>6s}"
)
for k in NAMES:
    row = f"{k:8s} {truth[k]:+9.5f}"
    for fit in (amplitude_fit, binflip_fit):
        value, error = fit[k]
        row += f" {value:+9.5f} ± {error:.5f} {(value - truth[k]) / error:+6.2f} "
    print(row)
print("r_b fitted / model:", np.round(r_fit / r_model, 3))